In [1]:
"""
STEP 2: VQE untuk H₂O — Output Grafik
=======================================
Menghitung ground state energy H₂O pakai VQE
dan scan PES sepanjang O-H stretch.
Output: semua dalam bentuk grafik!
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# BAGIAN 1: VQE untuk H₂O — Satu Geometri
# ============================================================

from pyscf import gto, scf, fci
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit_algorithms.utils import algorithm_globals
from qiskit.primitives import StatevectorEstimator as Estimator

algorithm_globals.random_seed = 42
np.random.seed(42)

# --- Geometri H₂O equilibrium ---
# O di origin, dua H di kanan-kiri
# Bond length O-H = 0.96 Å (equilibrium)
# Bond angle H-O-H = 104.5°

def build_h2o_geometry(r_OH):
    """
    Buat geometri H₂O dengan O-H bond length = r_OH (Angstrom)
    O di origin, sudut H-O-H = 104.5 derajat
    """
    angle = 104.5 * np.pi / 180  # konversi ke radian
    half_angle = angle / 2

    # Posisi dua atom H
    H1_x = r_OH * np.sin(half_angle)
    H1_y = r_OH * np.cos(half_angle)
    H2_x = -r_OH * np.sin(half_angle)
    H2_y = r_OH * np.cos(half_angle)

    return [
        ("O", (0.0,   0.0,   0.0)),
        ("H", (H1_x,  H1_y,  0.0)),
        ("H", (H2_x,  H2_y,  0.0)),
    ]

def run_vqe_h2o(r_OH, verbose=False):
    """
    Jalankan VQE untuk H₂O dengan bond length r_OH.
    Return: E_HF, E_VQE, E_FCI
    """
    geometry = build_h2o_geometry(r_OH)

    # --- PySCF untuk HF dan FCI ---
    mol = gto.Mole()
    mol.atom = geometry
    mol.basis = 'sto-3g'
    mol.charge = 0
    mol.spin = 0
    mol.unit = 'Angstrom'
    mol.verbose = 0
    mol.build()

    # Hartree-Fock
    mf = scf.RHF(mol)
    mf.verbose = 0
    mf.run()
    E_HF = mf.e_tot

    # FCI (exact reference)
    try:
        cisolver = fci.FCI(mf)
        cisolver.verbose = 0
        E_FCI, _ = cisolver.kernel()
    except:
        E_FCI = None

    # --- Qiskit Nature untuk VQE ---
    atom_str = f"O 0 0 0; H {geometry[1][1][0]:.4f} {geometry[1][1][1]:.4f} 0; H {geometry[2][1][0]:.4f} {geometry[2][1][1]:.4f} 0"

    driver = PySCFDriver(
        atom=atom_str,
        basis="sto-3g",
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM,
    )

    problem = driver.run()
    mapper = ParityMapper(num_particles=problem.num_particles)
    qubit_op = mapper.map(problem.second_q_ops()[0])

    hf_state = HartreeFock(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper,
    )

    ansatz = UCCSD(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper,
        initial_state=hf_state,
    )

    # Simpan history konvergensi
    energy_history = []
    def callback(eval_count, params, energy, meta):
        energy_history.append(energy)

    estimator = Estimator()
    optimizer = SLSQP(maxiter=300)

    vqe = VQE(
        estimator=estimator,
        ansatz=ansatz,
        optimizer=optimizer,
        initial_point=np.zeros(ansatz.num_parameters),
        callback=callback,
    )

    result = vqe.compute_minimum_eigenvalue(qubit_op)
    E_VQE = result.eigenvalue.real + problem.nuclear_repulsion_energy

    if verbose:
        print(f"  r(O-H) = {r_OH:.2f} Å | E_HF = {E_HF:.4f} | E_VQE = {E_VQE:.4f} | E_FCI = {E_FCI:.4f} Ha")

    return E_HF, E_VQE, E_FCI, energy_history, problem

# ============================================================
# BAGIAN 2: Hitung untuk satu geometri dulu (r_eq = 0.96 Å)
# ============================================================

print("Menghitung VQE H₂O di geometri equilibrium (r=0.96 Å)...")
r_eq = 0.96
E_HF_eq, E_VQE_eq, E_FCI_eq, history_eq, problem_eq = run_vqe_h2o(r_eq, verbose=True)
print("Done!")

# ============================================================
# BAGIAN 3: Scan PES — berbagai r_OH
# ============================================================

print("\nScan PES H₂O (O-H stretch)...")
r_values = np.arange(0.7, 2.51, 0.1)  # 0.7 sampai 2.5 Å

E_HF_list = []
E_VQE_list = []
E_FCI_list = []

for i, r in enumerate(r_values):
    print(f"  [{i+1}/{len(r_values)}] r = {r:.2f} Å...", end=' ')
    try:
        E_HF, E_VQE, E_FCI, _, _ = run_vqe_h2o(r)
        E_HF_list.append(E_HF)
        E_VQE_list.append(E_VQE)
        E_FCI_list.append(E_FCI)
        print(f"E_VQE = {E_VQE:.4f} Ha ✓")
    except Exception as e:
        print(f"Error: {e}")
        E_HF_list.append(None)
        E_VQE_list.append(None)
        E_FCI_list.append(None)

# Filter None values
valid = [(r, hf, vqe, fci_e) for r, hf, vqe, fci_e in
         zip(r_values, E_HF_list, E_VQE_list, E_FCI_list)
         if None not in (hf, vqe, fci_e)]

r_valid = np.array([v[0] for v in valid])
E_HF_valid = np.array([v[1] for v in valid])
E_VQE_valid = np.array([v[2] for v in valid])
E_FCI_valid = np.array([v[3] for v in valid])

print(f"\nSelesai! {len(valid)}/{len(r_values)} titik berhasil dihitung.")

# ============================================================
# BAGIAN 4: PLOT SEMUA GRAFIK
# ============================================================

fig = plt.figure(figsize=(16, 14))
fig.patch.set_facecolor('#0f0f1a')
gs = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

DARK_BG  = '#0f0f1a'
CARD_BG  = '#1a1a2e'
BLUE     = '#4fc3f7'
GREEN    = '#69f0ae'
ORANGE   = '#ffb74d'
RED      = '#ef5350'
PURPLE   = '#ce93d8'
WHITE    = '#e0e0e0'
GRAY     = '#555577'

def style_ax(ax, title):
    ax.set_facecolor(CARD_BG)
    ax.set_title(title, color=WHITE, fontsize=11, pad=10)
    ax.tick_params(colors=WHITE)
    ax.xaxis.label.set_color(WHITE)
    ax.yaxis.label.set_color(WHITE)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRAY)
    ax.grid(alpha=0.15, color=WHITE)

# ─── GRAFIK 1: Struktur Molekul H₂O ───────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(CARD_BG)
ax1.set_title('Struktur Molekul H₂O\n(Geometri Equilibrium)', color=WHITE, fontsize=11, pad=10)

geo = build_h2o_geometry(r_eq)
O_pos  = np.array(geo[0][1][:2])
H1_pos = np.array(geo[1][1][:2])
H2_pos = np.array(geo[2][1][:2])

# Gambar bond
for H_pos in [H1_pos, H2_pos]:
    ax1.plot([O_pos[0], H_pos[0]], [O_pos[1], H_pos[1]],
             color=WHITE, linewidth=3, zorder=1)

# Gambar atom
ax1.scatter(*O_pos,  s=800,  color=RED,   zorder=3, edgecolors=WHITE, linewidth=2)
ax1.scatter(*H1_pos, s=400,  color=BLUE,  zorder=3, edgecolors=WHITE, linewidth=2)
ax1.scatter(*H2_pos, s=400,  color=BLUE,  zorder=3, edgecolors=WHITE, linewidth=2)

# Label atom
ax1.text(*O_pos,   'O',  ha='center', va='center', color=WHITE, fontweight='bold', fontsize=12)
ax1.text(*H1_pos,  'H',  ha='center', va='center', color=WHITE, fontweight='bold', fontsize=10)
ax1.text(*H2_pos,  'H',  ha='center', va='center', color=WHITE, fontweight='bold', fontsize=10)

# Anotasi bond length dan angle
mid1 = (O_pos + H1_pos) / 2
mid2 = (O_pos + H2_pos) / 2
ax1.text(mid1[0]+0.05, mid1[1], f'r={r_eq}Å', color=GREEN, fontsize=9)
ax1.text(mid2[0]-0.25, mid2[1], f'r={r_eq}Å', color=GREEN, fontsize=9)
ax1.text(0, 0.15, '104.5°', color=ORANGE, fontsize=9, ha='center')

# Reaction coordinate arrow
ax1.annotate('', xy=(H1_pos[0]+0.3, H1_pos[1]+0.2),
             xytext=(H1_pos[0], H1_pos[1]),
             arrowprops=dict(arrowstyle='->', color=PURPLE, lw=2))
ax1.text(H1_pos[0]+0.35, H1_pos[1]+0.2, 'r(O-H)\nbertambah', color=PURPLE, fontsize=8)

ax1.set_xlim(-1.5, 1.8)
ax1.set_ylim(-0.3, 1.5)
ax1.set_aspect('equal')
ax1.axis('off')

# Legend
patches = [
    mpatches.Patch(color=RED,  label='Oksigen (O)'),
    mpatches.Patch(color=BLUE, label='Hidrogen (H)'),
]
ax1.legend(handles=patches, loc='lower right',
           facecolor=DARK_BG, edgecolor=GRAY, labelcolor=WHITE, fontsize=8)

# ─── GRAFIK 2: Konvergensi VQE ─────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
style_ax(ax2, 'Konvergensi VQE H₂O\n(Energi vs Iterasi Optimizer)')

if history_eq:
    iters = np.arange(1, len(history_eq) + 1)
    ax2.plot(iters, history_eq, color=BLUE, linewidth=2, label='VQE energy per iterasi')
    ax2.axhline(y=E_VQE_eq - problem_eq.nuclear_repulsion_energy,
                color=GREEN, linestyle='--', linewidth=2, label=f'Konvergen = {E_VQE_eq:.4f} Ha')
    ax2.axhline(y=E_HF_eq - problem_eq.nuclear_repulsion_energy,
                color=ORANGE, linestyle=':', linewidth=2, label=f'E_HF = {E_HF_eq:.4f} Ha')

    # Shaded area = penurunan dari HF ke VQE
    ax2.fill_between(iters,
                     E_HF_eq - problem_eq.nuclear_repulsion_energy,
                     history_eq,
                     alpha=0.15, color=GREEN, label='Korelasi elektron\nyang ditangkap')

ax2.set_xlabel('Iterasi Optimizer')
ax2.set_ylabel('Energi Elektronik (Ha)')
ax2.legend(facecolor=DARK_BG, edgecolor=GRAY, labelcolor=WHITE, fontsize=8)

# ─── GRAFIK 3: Perbandingan HF vs VQE vs FCI ───────────────
ax3 = fig.add_subplot(gs[1, 0])
style_ax(ax3, f'Perbandingan Energi H₂O\n(r_OH = {r_eq} Å)')

methods  = ['Hartree-\nFock', 'VQE\n(UCCSD)', 'FCI\n(Exact)']
energies = [E_HF_eq, E_VQE_eq, E_FCI_eq]
colors   = [ORANGE, BLUE, GREEN]

bars = ax3.bar(methods, energies, color=colors, alpha=0.8,
               edgecolor=WHITE, linewidth=1.2, width=0.5)

ax3.axhline(y=E_FCI_eq, color=GREEN, linestyle='--', alpha=0.5)

for bar, e in zip(bars, energies):
    ax3.text(bar.get_x() + bar.get_width()/2,
             e + 0.005,
             f'{e:.4f} Ha',
             ha='center', va='bottom', color=WHITE, fontsize=9)

# Anotasi selisih HF → VQE
ax3.annotate('',
    xy=(1, E_VQE_eq), xytext=(1, E_HF_eq),
    arrowprops=dict(arrowstyle='<->', color=PURPLE, lw=2))
ax3.text(1.27, (E_HF_eq + E_VQE_eq)/2,
         f'Korelasi\n= {abs(E_VQE_eq-E_HF_eq)*1000:.1f} mHa',
         color=PURPLE, fontsize=8, va='center')

ax3.set_ylabel('Total Energy (Ha)')

# ─── GRAFIK 4: Error vs FCI ────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
style_ax(ax4, 'Error vs FCI (Exact)\n[lebih kecil = lebih akurat]')

if E_FCI_eq is not None:
    err_HF  = abs(E_HF_eq  - E_FCI_eq) * 1000  # mHa
    err_VQE = abs(E_VQE_eq - E_FCI_eq) * 1000

    bars2 = ax4.bar(['Hartree-Fock', 'VQE (UCCSD)'],
                    [err_HF, err_VQE],
                    color=[ORANGE, BLUE],
                    alpha=0.8, edgecolor=WHITE, linewidth=1.2, width=0.4)

    ax4.axhline(y=1.6, color=RED, linestyle='--', linewidth=2,
                label='Chemical accuracy\n(1.6 mHa)')

    for bar, err in zip(bars2, [err_HF, err_VQE]):
        ax4.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'{err:.2f} mHa',
                 ha='center', va='bottom', color=WHITE, fontsize=10)

    ax4.set_ylabel('|E - E_FCI| (mHa)')
    ax4.set_yscale('log')
    ax4.legend(facecolor=DARK_BG, edgecolor=GRAY, labelcolor=WHITE)

    # Anotasi chemical accuracy zone
    ax4.fill_between([-0.5, 1.5], 0, 1.6, alpha=0.1, color=GREEN)
    ax4.text(0.5, 0.5, '✅ Chemical\nAccuracy Zone',
             ha='center', color=GREEN, fontsize=8)

# ─── GRAFIK 5: PES H₂O (O-H stretch) ──────────────────────
ax5 = fig.add_subplot(gs[2, :])
style_ax(ax5, 'Potential Energy Surface H₂O — O-H Stretch\n[Ini yang jadi input Trotter/SOFT untuk simulasi tunneling]')

if len(r_valid) > 0:
    ax5.plot(r_valid, E_HF_valid,  color=ORANGE, linewidth=2,
             marker='o', markersize=5, label='Hartree-Fock')
    ax5.plot(r_valid, E_VQE_valid, color=BLUE,   linewidth=2.5,
             marker='s', markersize=5, label='VQE (UCCSD)')
    ax5.plot(r_valid, E_FCI_valid, color=GREEN,  linewidth=2,
             linestyle='--', marker='^', markersize=5, label='FCI (Exact)')

    # Tandai titik minimum (equilibrium)
    min_idx = np.argmin(E_VQE_valid)
    ax5.scatter(r_valid[min_idx], E_VQE_valid[min_idx],
                s=200, color=RED, zorder=5, label=f'Minimum (r={r_valid[min_idx]:.2f}Å)')
    ax5.axvline(x=r_valid[min_idx], color=RED, linestyle=':', alpha=0.5)

    # Anotasi region
    ax5.axvspan(r_valid[0],  r_valid[min_idx],  alpha=0.05, color=BLUE,   label='O-H terikat (sumur)')
    ax5.axvspan(r_valid[min_idx], r_valid[-1], alpha=0.05, color=PURPLE, label='O-H memanjang')

    ax5.set_xlabel('r(O-H) — Reaction Coordinate (Å)', fontsize=11)
    ax5.set_ylabel('Ground State Energy (Ha)',           fontsize=11)
    ax5.legend(facecolor=DARK_BG, edgecolor=GRAY, labelcolor=WHITE,
               fontsize=9, ncol=3)

    # Anotasi penting
    ax5.text(0.75, E_VQE_valid[0]+0.1,
             'H dekat O\n(terikat kuat)',
             color=BLUE, fontsize=9, ha='center')
    ax5.text(2.2, E_VQE_valid[-1]-0.1,
             'H jauh dari O\n(hampir lepas)',
             color=PURPLE, fontsize=9, ha='center')

# ─── Title keseluruhan ─────────────────────────────────────
fig.suptitle('VQE H₂O — Ground State Energy & Potential Energy Surface',
             fontsize=14, color=WHITE, y=0.98)

plt.savefig('/mnt/user-data/outputs/step2_vqe_h2o.png',
            dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()

print("\n✅ Plot disimpan!")
print("\nCATATAN:")
print("  - Grafik 1 : Struktur H₂O dan reaction coordinate")
print("  - Grafik 2 : Konvergensi VQE (iterasi optimizer)")
print("  - Grafik 3 : Perbandingan energi HF vs VQE vs FCI")
print("  - Grafik 4 : Error vs FCI dalam mHa")
print("  - Grafik 5 : PES H₂O sepanjang O-H stretch")
print("\nNext: Step 3 → VQE untuk Glisin + PES double well")

ModuleNotFoundError: No module named 'qiskit_nature'